# Week 2 — LLM Loading & Inference Test
**Project:** AdaptivePromptGuard (APG)  
**Goal:** Load each 7B model with 4-bit quantization and verify inference works within VRAM budget.

Models tested:
- Mistral-7B-Instruct-v0.2
- Zephyr-7B-beta
- LLaMA-2-7B-Chat

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from src.model_loader import load_model, build_pipeline, generate_response, format_prompt
from src.utils import setup_logger, set_seed
from src.config import SYSTEM_PROMPT_DEFAULT

setup_logger()
set_seed(42)

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## Step 1 — Load Mistral-7B-Instruct (Start Here)
We start with Mistral because it has the most permissive license and doesn't require HuggingFace gated access.

In [ ]:
# Load Mistral
tokenizer, model = load_model('mistral', quantization='4bit')
pipe = build_pipeline(tokenizer, model)

vram_used = torch.cuda.memory_allocated() / 1e9
print(f'\nVRAM used after loading Mistral: {vram_used:.2f} GB')

In [ ]:
# Test Mistral inference
test_prompt = 'Explain what a large language model is in two sentences.'
formatted = format_prompt('mistral', test_prompt, system_prompt=SYSTEM_PROMPT_DEFAULT)

print('Prompt sent to model:')
print(formatted)
print('\n' + '-'*50)

response = generate_response(pipe, formatted)
print('Model Response:')
print(response)

In [ ]:
# IMPORTANT: Free VRAM before loading next model
import gc

del model, tokenizer, pipe
gc.collect()
torch.cuda.empty_cache()

vram_free = torch.cuda.memory_allocated() / 1e9
print(f'VRAM after cleanup: {vram_free:.2f} GB -- ready for next model')

## Step 2 — Load Zephyr-7B-beta

In [ ]:
tokenizer, model = load_model('zephyr', quantization='4bit')
pipe = build_pipeline(tokenizer, model)

vram_used = torch.cuda.memory_allocated() / 1e9
print(f'VRAM used after loading Zephyr: {vram_used:.2f} GB')

In [ ]:
test_prompt = 'What are the ethical concerns around large language models?'
formatted = format_prompt('zephyr', test_prompt, system_prompt=SYSTEM_PROMPT_DEFAULT)

response = generate_response(pipe, formatted)
print('Zephyr Response:')
print(response)

In [ ]:
# Cleanup
del model, tokenizer, pipe
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM cleared: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Step 3 — Load LLaMA-2-7B-Chat
> **Note:** LLaMA-2 requires HuggingFace gated access. You must:
> 1. Request access at: https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
> 2. Run `huggingface-cli login` and paste your token
> 
> If not approved yet, skip to the results summary below.

In [ ]:
# Only run this if you have HuggingFace access approved for LLaMA-2
try:
    tokenizer, model = load_model('llama2', quantization='4bit')
    pipe = build_pipeline(tokenizer, model)

    vram_used = torch.cuda.memory_allocated() / 1e9
    print(f'VRAM used after loading LLaMA-2: {vram_used:.2f} GB')

    test_prompt = 'What is the difference between supervised and unsupervised learning?'
    formatted = format_prompt('llama2', test_prompt, system_prompt=SYSTEM_PROMPT_DEFAULT)
    response = generate_response(pipe, formatted)
    print('LLaMA-2 Response:')
    print(response)

    del model, tokenizer, pipe
    gc.collect()
    torch.cuda.empty_cache()
except Exception as e:
    print(f'LLaMA-2 not available yet: {e}')
    print('Request access at: https://huggingface.co/meta-llama/Llama-2-7b-chat-hf')

## Results Summary — Week 2
Fill this in after running all three models.

In [ ]:
import pandas as pd

# Fill in your actual results below
results = {
    'Model':         ['Mistral-7B-Instruct', 'Zephyr-7B-beta', 'LLaMA-2-7B-Chat'],
    'Quantization':  ['4-bit', '4-bit', '4-bit'],
    'VRAM Used (GB)': [None, None, None],   # fill in from above cells
    'Load Time (s)':  [None, None, None],   # estimate
    'Status':         ['OK', 'OK', 'Pending access'],
}

df = pd.DataFrame(results)
print(df.to_string(index=False))